In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob
import os
import numpy as np
import h5py
import matplotlib.colors as colors
import random

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ..//util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from util.sim_data_helpers import get_data_from_snap_folder, get_data_from_header, get_cosmo_parameters

In [ ]:
def density_hist(path, snapN, resolution, _range=None):
    
    # get the data
    coordinates = get_data_from_snap_folder(path, snapN, "PartType1", "Coordinates")
    
    box_size = get_data_from_header(path, snapN, 'BoxSize') # in kpc
    dm_particle_mass = get_data_from_header(path, snapN, 'MassTable')[1]*1e10  # in M_sun
    physical_bin_volume = ((box_size/1e3)/resolution)**2 * (box_size/1e3)  # this is a projection, so one axis has the full length of the box

    unit_density_of_bin = dm_particle_mass/physical_bin_volume
    
    if _range is None:
        _range = [[0, box_size], [0, box_size]]
    
    # calculate the hist
    h, xedges, yedges = np.histogram2d(coordinates[:, 0], coordinates[:, 1], bins=resolution, range=_range)

    h = h*unit_density_of_bin # convert to actual density (M_sun/Mpc)
    
    return h, xedges, yedges

In [ ]:
def neutral_hydrogen_hist(path, snapN, resolution, little_h, _range=None):

    # load gas data
    coordinates = get_data_from_snap_folder(path, snapN, "PartType0", "Coordinates")
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    box_size = get_data_from_header(path, snapN, "BoxSize")  # ckpc/h

    physical_bin_volume = ((box_size/1e3)/resolution)**2 * (box_size/1e3)

    if _range is None:
        _range = [[0, box_size], [0, box_size]]

    # mass-weighted histogram
    h, xedges, yedges = np.histogram2d(
        coordinates[:,0],
        coordinates[:,1],
        bins=resolution,
        range=_range,
        weights=HI_mass
    )

    # convert to projected density
    h = h / physical_bin_volume

    return h, xedges, yedges

In [ ]:
gp_numbers_to_use = [i for i in range(0, 25)]
gp_indices = [i for i in range(len(gp_numbers_to_use))]
random.shuffle(gp_indices)  # randomize which boxes are shown
gp_indices = gp_indices[:20]

base_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/"
snapN = 12

sort_param = []
for gp_index in gp_indices:
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"

    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    sort_param.append((OmegaBaryon, gp_index))

_, sorted_indices = zip(*sorted(sort_param))

In [ ]:
sorted_indices

In [ ]:
fig, ax = plt.subplots(figsize=(20,20),dpi=500)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

gs = gridspec.GridSpec(5, 5, height_ratios=[1, 1, 1, 1, 0.05])
axes = []

norm_path = base_path + f"gridpoint{gp_numbers_to_use[sorted_indices[-1]]}/"

norm_h, xedges_ref, yedges_ref = density_hist(norm_path, snapN, 1024)
norm_h = norm_h/norm_h.mean()

abs_max = max(abs(norm_h.min()), abs(norm_h.max()))

# vmin = norm_h[norm_h != 0].min()

norm = colors.LogNorm(vmin=1/norm_h.max(), vmax=norm_h.max())

for i, gp_index in enumerate(sorted_indices): 
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"
    
    row = i//5
    col = i%5
    
    ax = fig.add_subplot(gs[row, col])

    h, xedges, yedges = density_hist(path, snapN, 1024)
    
    h = h/h.mean()
    
    im = ax.imshow(h.T, origin='lower', cmap='vanimo', norm=norm,
                extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
    
    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    
    ax.set_title(r"$\Omega_m$="+f"{Omega0:.2f}"+r"; $\Omega_\Lambda$="+f"{OmegaLambda:.2f}"+r"; h="+f"{HubbleParam:.2f}")
    if col == 0:
        ax.set_ylabel("y [ckpc/h]")
    else:
        ax.tick_params('y', labelleft=False)
    
    ax.set_xlabel("x [ckpc/h]")
    ax.set_facecolor('black')
    axes.append(ax)

cbar_ax = fig.add_subplot(gs[4, :])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r"$\delta$+1")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("plots/overdensity_grid.pdf", format="PDF")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20,20),dpi=500)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

gs = gridspec.GridSpec(5, 5, height_ratios=[1, 1, 1, 1, 0.05])
axes = []

norm_path = base_path + f"gridpoint{gp_numbers_to_use[sorted_indices[-1]]}/"

norm_h, xedges_ref, yedges_ref = density_hist(norm_path, snapN, 1024)

vmin = norm_h[norm_h != 0].min()

norm = colors.LogNorm(vmin=vmin, vmax=norm_h.max())

for i, gp_index in enumerate(sorted_indices): 
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"
    
    row = i//5
    col = i%5
    
    ax = fig.add_subplot(gs[row, col])

    h, xedges, yedges = density_hist(path, snapN, 1024)
    
    im = ax.imshow(h.T, origin='lower', cmap='magma', norm=norm,
                extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
    
    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    
    ax.set_title(r"$\Omega_m$="+f"{Omega0:.2f}"+r"; $\Omega_\Lambda$="+f"{OmegaLambda:.2f}"+r"; h="+f"{HubbleParam:.2f}")
    if col == 0:
        ax.set_ylabel("y [ckpc/h]")
    else:
        ax.tick_params('y', labelleft=False)
    
    ax.set_xlabel("x [ckpc/h]")
    ax.set_facecolor('black')
    axes.append(ax)

cbar_ax = fig.add_subplot(gs[4, :])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r"Density [$M_\odot$ / Mpc]")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("plots/Density_grid.pdf", format="PDF")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20,20),dpi=500)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

gs = gridspec.GridSpec(5, 5, height_ratios=[1, 1, 1, 1, 0.05])
axes = []

reference_path = "/vera/u/jerbo/my_ptmp/L25n256_suite/reference_point/"
norm_path = base_path + f"gridpoint{gp_numbers_to_use[sorted_indices[6]]}/"

reference_h, xedges_ref, yedges_ref  = density_hist(reference_path, snapN, 1024)
reference_h = reference_h * 1e-9 # convert from M_sun/Mpc^3 to M_sun/kpc^3
norm_h, *_ = density_hist(norm_path, snapN, 1024)
norm_h = norm_h * 1e-9 # convert from M_sun/Mpc^3 to M_sun/kpc^3
abs_max = max(abs(norm_h.min()), abs(norm_h.max()))
norm = colors.SymLogNorm(linthresh=1e2, vmin=-abs_max, vmax=abs_max, base=10)

for i, gp_index in enumerate(sorted_indices): 
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"
    
    row = i//5
    col = i%5
    
    ax = fig.add_subplot(gs[row, col])

    h, xedges, yedges = density_hist(path, snapN, 1024)

    h = h * 1e-9 # convert from M_sun/Mpc^3 to M_sun/kpc^3

    h = (h - reference_h) 
    
    im = ax.imshow(h.T, origin='lower', cmap='vanimo', norm=norm,
                extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
    
    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    
    ax.set_title(r"$\Omega_m$="+f"{Omega0:.2f}"+r"; $\Omega_\Lambda$="+f"{OmegaLambda:.2f}"+r"; h="+f"{HubbleParam:.2f}")
    if col == 0:
        ax.set_ylabel("y [ckpc/h]")
    else:
        ax.tick_params('y', labelleft=False)
    
    ax.set_xlabel("x [ckpc/h]")
    ax.set_facecolor('black')
    axes.append(ax)

cbar_ax = fig.add_subplot(gs[4, :])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r"$\Delta$ Density [$M_\odot$/ckpc]")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("plots/Delta_density_grid.pdf", format="PDF")
plt.show()

In [ ]:
data = {}

for gp_index in sorted_indices:
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"

    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    h, xedges, yedges = neutral_hydrogen_hist(path, snapN, 1024, little_h=HubbleParam)

    data[gp] = {
        "h": h,
        "xedges": xedges,
        "yedges": yedges,
        "Omega0": Omega0,
        "OmegaLambda": OmegaLambda,
        "HubbleParam": HubbleParam,
        "Omegab": OmegaBaryon
    }

# reference normalization
gp_norm = gp_numbers_to_use[sorted_indices[-1]]
norm_h = data[gp_norm]["h"]
vmin = norm_h[norm_h != 0].min()
vmax = norm_h.max()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8), dpi=500)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

gs = gridspec.GridSpec(5, 5, height_ratios=[1,1,1,1,0.07])
axes = []

norm = colors.LogNorm(vmin=vmin, vmax=vmax)

for i, gp_index in enumerate(sorted_indices):

    gp = gp_numbers_to_use[gp_index]
    d = data[gp]

    row = i//5
    col = i%5
    ax = fig.add_subplot(gs[row, col], sharex=axes[0] if axes else None)

    im = ax.imshow(
        d["h"].T,
        origin="lower",
        cmap="magma",
        norm=norm,
        interpolation='none',
        extent=[d["xedges"][0], d["xedges"][-1],
                d["yedges"][0], d["yedges"][-1]]
    )
    ax.set_aspect('equal')
    if i == 0:
        bar_length = 20000

        x0 = d["xedges"][0] + 0.1 * (d["xedges"][-1] - d["xedges"][0])
        y0 = d["yedges"][0] + 0.05 * (d["yedges"][-1] - d["yedges"][0])

        ax.plot([x0, x0 + bar_length], [y0, y0], color="white", lw=2)

        ax.text(
            x0 + bar_length/2,
            y0 + 0.02*(d["yedges"][-1]-d["yedges"][0]),
            "20 cMpc",
            color="white",
            ha="center",
            va="bottom",
            fontsize=8
        )

    ax.text(
    0.97, 0.97,
    r"$\Omega_b$="+f"{d['Omegab']:.2f}"
    + r" $\Omega_\Lambda$="+f"{d['OmegaLambda']:.2f}\n"
    + r"$\Omega_m$="+f"{d['Omega0']:.2f}"
    + r"  $h$="+f"{d['HubbleParam']:.2f}",
    transform=ax.transAxes,
    color="white",
    fontsize=7,
    verticalalignment="top",
    horizontalalignment="right",
    #bbox=dict(facecolor="black", alpha=0.3, edgecolor="none", pad=2)
    )

    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.set_facecolor('black')
    axes.append(ax)

cbar_ax = fig.add_subplot(gs[4, :])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r"HI density [$M_\odot$ / cMpc$^3$]")

plt.tight_layout(rect=[0,0.1,1,1])
plt.savefig("plots/HI_Density_grid.pdf", format="PDF")
plt.show()

## Plot one box 

In [ ]:
path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/reference/"
snapN = 12
HubbleParam = 0.6774

h, xedges, yedges = density_hist(path, snapN, 1024)#, little_h=HubbleParam)

z = get_data_from_header(path, snapN, "Redshift")

In [ ]:
fig, ax = plt.subplots()

norm = colors.LogNorm(vmin=1e10, vmax=1e13)

im = ax.imshow(
        h.T,
        origin="lower",
        cmap="magma",
        norm=norm,
        interpolation='none',
        extent=[xedges[0], xedges[-1],
                yedges[0], yedges[-1]]
    )

bar_length = 10000

x0 = xedges[0] + 0.1 * (xedges[-1] - xedges[0])
y0 = yedges[0] + 0.05 * (yedges[-1] - yedges[0])

ax.plot([x0, x0 + bar_length], [y0, y0], color="white", lw=2)

ax.text(
x0 + bar_length/2,
y0 + 0.02*(yedges[-1]-yedges[0]),
"10 cMpc",
color="white",
ha="center",
va="bottom",
fontsize=10,
weight='bold'
)

ax.text(
0.97, 0.97,
f"$z = {z:.2f}$",
transform=ax.transAxes,
color="white",
fontsize=10,
verticalalignment="top",
horizontalalignment="right",
)

ax.set_yticklabels([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_xticks([])
ax.set_facecolor('black')

cbar = fig.colorbar(im)
cbar.set_label(r"HI density [$M_\odot$ / cMpc$^3$]")

plt.savefig("plots/box_visualization_L50n512_density.pdf", format="pdf")
# plt.show()